# QnA Bot Pipeline: FAISS + Sentence Transformers

This notebook demonstrates the complete pipeline for building a semantic search-based QnA bot:
1. **Load & Preprocess** - Load restaurant Q&A data
2. **Embed** - Convert text to vectors using HuggingFace MiniLM
3. **Index** - Store vectors in FAISS for fast similarity search
4. **Search** - Retrieve answers using semantic similarity
5. **Evaluate** - Test with example queries

---

## Section 1: Setup & Imports

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

print("✓ All imports successful!")

## Section 2: Load & Explore Dataset

Load the restaurant Q&A dataset from JSON file.

In [ ]:
# Load JSON data
data_path = Path("data/restaurant_qna.json")
with open(data_path, "r") as f:
    data = json.load(f)

qa_pairs = data["qa_pairs"]
df = pd.DataFrame(qa_pairs)

print(f"Loaded {len(df)} Q&A pairs")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDataset shape: {df.shape}")

In [ ]:
# Display first few rows
df.head(3)

## Section 3: Prepare Text for Embedding

Combine questions and answers for richer semantic representation.

In [ ]:
# Combine question and answer
df['combined_text'] = df['question'] + " " + df['answer']

texts = df['combined_text'].tolist()
questions = df['question'].tolist()
answers = df['answer'].tolist()

print(f"Prepared {len(texts)} texts for embedding")
print(f"\nExample combined text:")
print(f"  {texts[0][:100]}...")

## Section 4: Generate Embeddings

Convert text to 384-dimensional vectors using Sentence Transformers MiniLM-L6-v2 model.

**Why MiniLM?**
- Small (22M parameters) but powerful
- Fast inference
- Trained on semantic similarity tasks
- 384-dimensional embeddings

In [ ]:
# Load the embedding model
print("Loading Sentence Transformer model (MiniLM-L6-v2)...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"✓ Model loaded successfully")

In [ ]:
# Generate embeddings
print("Generating embeddings for all Q&A pairs...")
embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

print(f"\n✓ Embeddings generated!")
print(f"  Shape: {embeddings.shape}")
print(f"  Type: {embeddings.dtype}")
print(f"\nEmbedding statistics:")
print(f"  Mean: {embeddings.mean():.4f}")
print(f"  Std: {embeddings.std():.4f}")
print(f"  Min: {embeddings.min():.4f}")
print(f"  Max: {embeddings.max():.4f}")

## Section 5: Create FAISS Index

Store embeddings in FAISS (Facebook AI Similarity Search) for fast nearest-neighbor retrieval.

**Why FAISS?**
- Ultra-fast similarity search
- Scales to millions of vectors
- Multiple index types (L2, IP, HNSW, etc.)
- Production-ready

In [ ]:
# Create FAISS index
print("Creating FAISS index...")
dimension = embeddings.shape[1]
print(f"  Vector dimension: {dimension}")

# Use IndexFlatL2 for L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)

# Add vectors to index
index.add(embeddings)

print(f"\n✓ FAISS index created!")
print(f"  Total vectors: {index.ntotal}")
print(f"  Index type: IndexFlatL2")

## Section 6: Test Semantic Search

Search for relevant Q&A pairs using natural language queries.

In [ ]:
def search_qa(query, top_k=3):
    """Search for relevant Q&A pairs"""
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True)
    
    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)
    
    # Format results
    results = []
    for i, idx in enumerate(indices[0]):
        distance = float(distances[0][i])
        # Convert L2 distance to similarity (0-1 scale)
        similarity = 1 / (1 + distance)
        
        results.append({
            'rank': i + 1,
            'question': questions[idx],
            'answer': answers[idx],
            'distance': distance,
            'similarity': similarity
        })
    
    return results

print("✓ Search function defined")

In [ ]:
# Test Query 1
query = "What are your opening hours?"
print(f"Query: '{query}'")
print("=" * 80)
results = search_qa(query, top_k=3)

for r in results:
    print(f"\n[Rank {r['rank']}] Similarity: {r['similarity']:.4f}")
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer'][:100]}...")

In [ ]:
# Test Query 2
query = "Do you deliver food?"
print(f"\nQuery: '{query}'")
print("=" * 80)
results = search_qa(query, top_k=3)

for r in results:
    print(f"\n[Rank {r['rank']}] Similarity: {r['similarity']:.4f}")
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer'][:100]}...")

In [ ]:
# Test Query 3
query = "Do you have vegetarian options?"
print(f"\nQuery: '{query}'")
print("=" * 80)
results = search_qa(query, top_k=3)

for r in results:
    print(f"\n[Rank {r['rank']}] Similarity: {r['similarity']:.4f}")
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer'][:100]}...")

## Section 7: Save Index and Model Metadata

Save the FAISS index and metadata for use in the Flask app.

In [ ]:
# Create models directory
models_path = Path("models")
models_path.mkdir(exist_ok=True)

# Save FAISS index
faiss.write_index(index, str(models_path / "faiss_index.bin"))
print(f"✓ FAISS index saved to: models/faiss_index.bin")

# Save metadata
metadata = {
    "questions": questions,
    "answers": answers,
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "dimension": dimension,
    "total_vectors": index.ntotal
}

with open(models_path / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to: models/metadata.json")
print(f"\nModel Info:")
print(f"  Total Q&A pairs: {len(questions)}")
print(f"  Vector dimension: {dimension}")
print(f"  Embedding model: sentence-transformers/all-MiniLM-L6-v2")

## Section 8: Summary

✅ **Pipeline Complete!**

You have successfully built a semantic QnA bot:
1. ✓ Loaded and explored restaurant Q&A dataset
2. ✓ Generated embeddings using MiniLM-L6-v2
3. ✓ Created FAISS index for fast similarity search
4. ✓ Tested semantic search with multiple queries
5. ✓ Saved model and metadata for Flask deployment

### Next Steps:
1. Run `python app.py` to start the Flask server
2. Open http://127.0.0.1:5000 in your browser
3. Try asking questions about the restaurant

### Pipeline Architecture:
```
Q&A Dataset (JSON)
        ↓
Text Preprocessing
        ↓
Sentence Transformers (MiniLM)
        ↓
Embeddings (384-dim vectors)
        ↓
FAISS Index (Flat L2)
        ↓
Saved for Flask App
        ↓
Web UI (HTML/JS)
```